## Rule analysis and correction mechanism

I compare Limited and Casco only within the same variant and deductible. Comparing `casco_compact_500` (620) with `limited_casco_premium_100` (1100) would mix cover levels. MTPL has one base price, compared with the lowest price at the next level.

The example has 11 violations across 24 comparisons: all 11 are in 12 Limited–Casco comparisons, while all 12 MTPL–Limited comparisons pass. The pattern points to one block issue. Since 91.7% of the Limited–Casco block fails, the working 50% threshold triggers a block correction.

For this example, I chose to increase the prices of the Casco block. Lowering Limited would need claims-cost, margin or market evidence not provided.

I apply one pre-rounding factor to the whole Casco block. It is the largest `(Limited + 1) / Casco` ratio: 1.09467. The `+1` enforces strict integer ordering after rounding; using 1.0933 would make 750 round to 820, equal to Limited. A common factor preserves the existing Casco structure more closely than 11 separate corrections. For compact Casco, a local correction would change the deductible step from −17.3% to −20.7%; block scaling avoids that distortion.



In [1]:
example_prices_to_correct = {
    "mtpl": 400,
    "limited_casco_compact_100": 820,
    "limited_casco_compact_200": 760,
    "limited_casco_compact_500": 650,
    "limited_casco_basic_100": 900,
    "limited_casco_basic_200": 780,
    "limited_casco_basic_500": 600,
    "limited_casco_comfort_100": 950,
    "limited_casco_comfort_200": 870,
    "limited_casco_comfort_500": 720,
    "limited_casco_premium_100": 1100,
    "limited_casco_premium_200": 980,
    "limited_casco_premium_500": 800,
    "casco_compact_100": 750,
    "casco_compact_200": 700,
    "casco_compact_500": 620,
    "casco_basic_100": 830,
    "casco_basic_200": 760,
    "casco_basic_500": 650,
    "casco_comfort_100": 900,
    "casco_comfort_200": 820,
    "casco_comfort_500": 720,
    "casco_premium_100": 1050,
    "casco_premium_200": 950,
    "casco_premium_500": 780,
}

In [2]:
import math


def pricing_enforcement(prices: dict[str, int]) -> dict[str, int]:
    products = ["limited_casco", "casco"]
    variants = ["compact", "basic", "comfort", "premium"]
    deductibles = [100, 200, 500]
    descending = deductibles[::-1]
    # The order prevents the structure from being broken. First, the internal relations (deductible and variant) are adjusted, and then the product levels are processed (MTPL before Casco).
    rule_order = ["deductible", "variant", "product_level_mtpl", "product_level_casco"]
    higher_block = {"product_level_mtpl": "limited_casco", "product_level_casco": "casco"}
    constraints = []
    for product in products:
        for variant in variants:
            for cheaper_deductible, dearer_deductible in zip(descending, descending[1:]):
                constraints.append((f"{product}_{variant}_{cheaper_deductible}", f"{product}_{variant}_{dearer_deductible}", "deductible"))
        for deductible in deductibles:
            for entry in ["compact", "basic"]:
                constraints.append((f"{product}_{entry}_{deductible}", f"{product}_comfort_{deductible}", "variant"))
            constraints.append((f"{product}_comfort_{deductible}", f"{product}_premium_{deductible}", "variant"))
    for variant in variants:
        for deductible in deductibles:
            constraints.append(("mtpl", f"limited_casco_{variant}_{deductible}", "product_level_mtpl"))
            constraints.append((f"limited_casco_{variant}_{deductible}", f"casco_{variant}_{deductible}", "product_level_casco"))
    corrected = dict(prices)
    print(f"violations before: {sum(corrected[cheap] >= corrected[dear] for cheap, dear, _ in constraints)}")
    for _ in range(10):
        if not any(corrected[cheap] >= corrected[dear] for cheap, dear, _ in constraints):
            break
        for rule in rule_order:
            relevant = [triple for triple in constraints if triple[2] == rule]
            violated = [triple for triple in relevant if corrected[triple[0]] >= corrected[triple[1]]]
            if not violated:
                continue
            block = higher_block.get(rule)
            internal_valid = block is not None and all(
                corrected[cheap] < corrected[dear]
                for cheap, dear, other_rule in constraints
                if other_rule in ("deductible", "variant") and dear.startswith(block)
            )
            # A threshold of 50% separates a systemic block error from an individual anomaly. A block is moved only if the internal relations in it are already correct.
            if internal_valid and 2 * len(violated) >= len(relevant):
                # +1 is added to the numerator due to strict inequality. The standard ratio after rounding (750 * 1.0933) would give exactly 820, so the violation would remain.
                factor = max((corrected[cheap] + 1) / corrected[dear] for cheap, dear, _ in violated)
                print(f"{rule}: {len(violated)}/{len(relevant)} violated, block {block} x {factor:.6f}")
                for key in [f"{block}_{variant}_{deductible}" for variant in variants for deductible in deductibles]:
                    previous = corrected[key]
                    # Prices are integers; ceil keeps an uplift from rounding a corrected price back to equality.
                    corrected[key] = math.ceil(previous * factor)
                    print(f"  {key}: {previous} -> {corrected[key]}")
            else:
                print(f"{rule}: {len(violated)}/{len(relevant)} violated, local")
                for cheap, dear, _ in violated:
                    if corrected[dear] < corrected[cheap] + 1:
                        previous = corrected[dear]
                        corrected[dear] = corrected[cheap] + 1
                        print(f"  {dear}: {previous} -> {corrected[dear]}")
    remaining = sum(corrected[cheap] >= corrected[dear] for cheap, dear, _ in constraints)
    if remaining != 0:
        raise ValueError("correction did not converge")
    print(f"violations after: {remaining}")
    return corrected


In [3]:
corrected_prices = pricing_enforcement(example_prices_to_correct)

violations before: 11
product_level_casco: 11/12 violated, block casco x 1.094667
  casco_compact_100: 750 -> 821
  casco_compact_200: 700 -> 767
  casco_compact_500: 620 -> 679
  casco_basic_100: 830 -> 909
  casco_basic_200: 760 -> 832
  casco_basic_500: 650 -> 712
  casco_comfort_100: 900 -> 986
  casco_comfort_200: 820 -> 898
  casco_comfort_500: 720 -> 789
  casco_premium_100: 1050 -> 1150
  casco_premium_200: 950 -> 1040
  casco_premium_500: 780 -> 854
violations after: 0


In [4]:
print(pricing_enforcement(corrected_prices) == corrected_prices)

violations before: 0
violations after: 0
True


In [5]:
single_anomaly_prices = {
    "mtpl": 400,
    "limited_casco_compact_100": 800,
    "limited_casco_compact_200": 720,
    "limited_casco_compact_500": 650,
    "limited_casco_basic_100": 820,
    "limited_casco_basic_200": 740,
    "limited_casco_basic_500": 660,
    "limited_casco_comfort_100": 900,
    "limited_casco_comfort_200": 810,
    "limited_casco_comfort_500": 730,
    "limited_casco_premium_100": 1000,
    "limited_casco_premium_200": 900,
    "limited_casco_premium_500": 810,
    "casco_compact_100": 920,
    "casco_compact_200": 830,
    "casco_compact_500": 640,
    "casco_basic_100": 940,
    "casco_basic_200": 850,
    "casco_basic_500": 760,
    "casco_comfort_100": 1030,
    "casco_comfort_200": 930,
    "casco_comfort_500": 840,
    "casco_premium_100": 1150,
    "casco_premium_200": 1030,
    "casco_premium_500": 930,
}

single_anomaly_corrected = pricing_enforcement(single_anomaly_prices)

violations before: 1
product_level_casco: 1/12 violated, local
  casco_compact_500: 640 -> 651
violations after: 0
